# FAISS 向量检索（RAG 检索环节）

> 定位：FAISS 在 RAG 里扮演「向量库 + 检索」的角色，和 01-RAG.ipynb 里的 Chroma 是**并列的替代方案**

## 一、FAISS 是什么

- **FAISS** = Facebook AI Similarity Search，Meta 开源的高性能向量检索库
- 在 RAG 里的位置：文本 → 切分 → **向量化** → **FAISS 存储** → **相似度检索** → 生成答案
- 特点：
  - 检索**极快**，适合大规模向量（百万级、亿级）
  - 无内置 Web UI 和元数据过滤，需自己 `save_local` 持久化
  - 底层是 C++，LangChain 做了 Python 封装

## 二、FAISS vs Chroma 对比

| 维度 | FAISS | Chroma |
|------|-------|--------|
| 定位 | 高性能向量检索库 | 开箱即用的向量数据库 |
| 速度 | 极快，适合超大规模 | 中等，小中型够用 |
| 持久化 | `save_local` / `load_local` 手动管理 | 内置（`persist_directory`） |
| 元数据过滤 | 需自己实现 | 内置 `where` 过滤 |
| Web UI | 无 | 有 |
| 上手难度 | 略高（底层 API 多） | 简单 |
| 何时选它 | 数据量巨大、追求检索速度 | 快速原型、需要过滤和可视化 |

## 三、FAISS

### 1. 安装依赖

```bash
pip install faiss-cpu     # CPU 版（学习够用）
pip install faiss-gpu     # GPU 版（生产大规模用）
```

> ⚠️ 包名是 `faiss-cpu`，不是 `faiss`；安装后 import 时写 `import faiss`。

### 2. 准备数据（加载 + 切分）

In [18]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 文档加载
loader = TextLoader(r"../data/deepseek百度百科.txt", encoding='utf-8')
documents = loader.load()

# 2. 文档切分
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(documents)
print("切分后块数：", len(split_documents))

切分后块数： 58


### 3. 嵌入模型

把文本变成向量（和 01-RAG 用的是同一个阿里通义千问嵌入模型）。

In [9]:
import os
from langchain_community.embeddings import DashScopeEmbeddings

# 嵌入模型：文本 -> 向量
embeddings = DashScopeEmbeddings(
    model="text-embedding-v3",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),
)

## 4.1 建库方式一：高层 API（推荐）

`FAISS.from_documents` 一步建库，和 `Chroma.from_documents` 几乎一模一样。

In [3]:
from langchain_community.vectorstores import FAISS

# ⭐ 高层 API：一步完成「向量化 + 建索引 + 存文档」
vector_store = FAISS.from_documents(split_documents, embeddings)
print("高层 API 建库完成")

高层 API 建库完成


## 4.2 建库方式二：底层 API（手动控制 index）

想理解 FAISS 内部结构、或需要自定义索引类型时，用底层 API。

In [11]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

# ⭐ 底层 API 分三步：建索引 -> 组装 FAISS 对象 -> 写入文档
# 1. 先算向量维度（随便问一句，看它变成多少维）
vec_dim = len(embeddings.embed_query("faiss_index"))
print("向量维度：", vec_dim)

# 2. 创建索引：IndexFlatL2 = 暴力检索 + L2 距离（精确但慢，小规模够用）
index = faiss.IndexFlatL2(vec_dim)

# 3. 组装 FAISS 对象
vector_store = FAISS(
    embedding_function=embeddings,   # 嵌入模型
    index=index,                     # 索引
    docstore=InMemoryDocstore(),     # 内存文档存储（存 page_content + metadata）
    index_to_docstore_id={},         # 索引 id -> 文档 id 映射，新建可空
)

# 4. 将文档块加入FAISS
vector_store.add_documents(documents=split_documents)
print("底层 API 建库完成")

向量维度： 1024
底层 API 建库完成


### 5. 检索（相似度搜索）

In [19]:
# 相似度检索（带分数）：分数是 L2 距离，越小越相似
query = "DeepSeek对汽车行业有什么影响？"
similar_docs = vector_store.similarity_search_with_score(query, k=2)

for doc, score in similar_docs:
    print(f"== Score == : {score}")
    print("== doc.page_content ==:",doc.page_content[:100])
    print("-" * 100)

== Score == : 0.5904936790466309
== doc.page_content ==: 2025年2月7日，岚图汽车已完成与DeepSeek模型的深度融合。
2025年2月8日，宝骏汽车宣布其灵语智舱与DeepSeek大模型完成深度融合。同日消息，QuestMobile的最新数据显示， 
----------------------------------------------------------------------------------------------------
== Score == : 0.6117149591445923
== doc.page_content ==: DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据。由知名私募巨头幻方量化孕
----------------------------------------------------------------------------------------------------


### 6. 检索 + 生成（接入 RAG 链）

和 01-RAG 里 Chroma 的用法完全一致：`as_retriever()` 转检索器，接进链。

In [17]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# ⭐ FAISS 转检索器，对接 RAG 链
retriever = vector_store.as_retriever()

prompt_template = ChatPromptTemplate.from_messages([
    ("human", "仅使用提供的上下文回答下面的问题：{question} 上下文：{context}")
])
model = init_chat_model(model="deepseek-flash")

chain = {"question": RunnablePassthrough(), "context": retriever} | prompt_template | model

resp = chain.invoke("DeepSeek对汽车行业有什么影响？")
print(resp.content)

根据提供的上下文，DeepSeek对汽车行业的影响主要体现为与多家汽车企业的融合：

- 2025年2月7日，岚图汽车完成与DeepSeek模型的深度融合。
- 2025年2月8日，宝骏汽车宣布其灵语智舱与DeepSeek大模型完成深度融合。
- 2025年2月9日，上汽荣威飞凡与DeepSeek达成深度技术融合。
- 在汽车相关的电池领域，DeepSeek在电池知识问答和电池文本挖掘任务上表现优异，在电池设计任务上具备初步总结能力，但尚欠缺科学分析能力，仍需要垂直领域大模型解决。

除上述内容外，提供的上下文未提及其他汽车行业影响。


### 7. 持久化与合并

FAISS 不像 Chroma 自动持久化，要手动 `save_local` / `load_local`。

In [25]:
# ⭐ 保存索引到本地
vector_store.save_local(folder_path="../index/faiss_index", index_name="deepseek_baidu")

# 重新加载（allow_dangerous_deserialization=True 是必须的，否则报错）
new_store = FAISS.load_local(
    folder_path="../index/faiss_index",
    index_name="deepseek_baidu",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

query = "DeepSeek对汽车行业有什么影响？"
similar_docs = new_store.similarity_search_with_score(query, k=2)

for doc, score in similar_docs:
    print(f"== Score == : {score}")
    print("== doc.page_content ==:",doc.page_content[:100])
    print("-" * 100)



# ⭐ 合并两个索引（增量更新场景）
db1 = FAISS.from_texts(["A 文档内容"], embeddings)
db2 = FAISS.from_texts(["B 文档内容"], embeddings)
print("db1.docstore._dict:", db1.docstore._dict)
print("db2.docstore._dict:", db2.docstore._dict)

db1.merge_from(db2)
print("db1.docstore._dict:", db1.docstore._dict)

== Score == : 0.5904936790466309
== doc.page_content ==: 2025年2月7日，岚图汽车已完成与DeepSeek模型的深度融合。
2025年2月8日，宝骏汽车宣布其灵语智舱与DeepSeek大模型完成深度融合。同日消息，QuestMobile的最新数据显示， 
----------------------------------------------------------------------------------------------------
== Score == : 0.6117149591445923
== doc.page_content ==: DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据。由知名私募巨头幻方量化孕
----------------------------------------------------------------------------------------------------
db1.docstore._dict: {'a8bfdd48-6c3b-417d-aac0-f9727b1a96e5': Document(id='a8bfdd48-6c3b-417d-aac0-f9727b1a96e5', metadata={}, page_content='A 文档内容')}
db2.docstore._dict: {'48d10380-f8a3-428e-9a2e-7ff3be8fed4d': Document(id='48d10380-f8a3-428e-9a2e-7ff3be8fed4d', metadata={}, page_content='B 文档内容')}
db1.docstore._dict: {'a8bfdd48-6c3b-417d-aac0-f9727b1a96e5': Document(id='a8bfdd48-6c3b-417d-aac0-f9727b1a96e5', metadata={}, page_content='A 文档内容'), '48d10380-f8a3-428e-9a2e-7ff3be8

In [28]:
# new_store.merge_from(db1) # 存在索引冲突时会报错
new_store.similarity_search_with_score("A 文档内容", k=2)

[(Document(id='a8bfdd48-6c3b-417d-aac0-f9727b1a96e5', metadata={}, page_content='A 文档内容'),
  np.float32(0.0)),
 (Document(id='48d10380-f8a3-428e-9a2e-7ff3be8fed4d', metadata={}, page_content='B 文档内容'),
  np.float32(0.26135954))]

## 常见坑
- 包名是 `faiss-cpu` / `faiss-gpu`，import 时写 `import faiss`
- `load_local` 必须加 `allow_dangerous_deserialization=True`，否则抛安全异常
- 分数是 **L2 距离，越小越相似**（和 Chroma 一样），别当成相似度
- `IndexFlatL2` 是暴力精确检索，数据几十万以上要换 `IndexIVFFlat` / HNSW 等近似索引
- FAISS 无内置元数据过滤，需要过滤时优先用 Chroma